### RAG from Data Ingestion to Vector DB

Data Ingestion-->Data parsing(chunking)-->Embedding-->Vector DB

#### loading

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\lenovo\AppData\Local\Temp\ipykernel_25204\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\lenovo\GitProjects\LangChain-RAG-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Read all the pdf's inside a directory

def process_all_pdfs(pdf_directory):
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            #added 2 keys ('source_file' & 'file_type') extra to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 3 PDF files to process

Processing: n3.pdf
  ✓ Loaded 54 pages

Processing: n4.pdf
  ✓ Loaded 20 pages

Processing: n5.pdf
  ✓ Loaded 46 pages

Total documents loaded: 120


In [10]:
all_pdf_documents

[Document(metadata={'producer': 'Pdftools SDK', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2025-07-07T11:00:56+00:00', 'moddate': '2026-07-23T17:55:51+05:30', 'source': '..\\data\\pdf\\n3.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': 'n3.pdf', 'file_type': 'pdf'}, page_content='WEB ANALYTICS'),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2025-07-07T11:00:56+00:00', 'moddate': '2026-07-23T17:55:51+05:30', 'source': '..\\data\\pdf\\n3.pdf', 'total_pages': 54, 'page': 1, 'page_label': '2', 'source_file': 'n3.pdf', 'file_type': 'pdf'}, page_content='Introduction to www\n• The Web is a global set of documents, images and other resources, logically\ninterrelated by hyperlinks and referenced with Uniform Resource Identifiers (URIs).\n• URIs symbolically identify services, servers, and other databases, and the\ndocuments and resources that they can provide.\n• Hypertext Transfer Protocol (HTT

#### chunking

In [4]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
   
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 120 documents into 119 chunks

Example chunk:
Content: WEB ANALYTICS...
Metadata: {'producer': 'Pdftools SDK', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2025-07-07T11:00:56+00:00', 'moddate': '2026-07-23T17:55:51+05:30', 'source': '..\\data\\pdf\\n3.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': 'n3.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Pdftools SDK', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2025-07-07T11:00:56+00:00', 'moddate': '2026-07-23T17:55:51+05:30', 'source': '..\\data\\pdf\\n3.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': 'n3.pdf', 'file_type': 'pdf'}, page_content='WEB ANALYTICS'),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2025-07-07T11:00:56+00:00', 'moddate': '2026-07-23T17:55:51+05:30', 'source': '..\\data\\pdf\\n3.pdf', 'total_pages': 54, 'page': 1, 'page_label': '2', 'source_file': 'n3.pdf', 'file_type': 'pdf'}, page_content='Introduction to www\n• The Web is a global set of documents, images and other resources, logically\ninterrelated by hyperlinks and referenced with Uniform Resource Identifiers (URIs).\n• URIs symbolically identify services, servers, and other databases, and the\ndocuments and resources that they can provide.\n• Hypertext Transfer Protocol (HTT

#### embedding

In [16]:
pip install senetence_tranformers

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement senetence_tranformers (from versions: none)
ERROR: No matching distribution found for senetence_tranformers

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        #model_name: HuggingFace model name for sentence embeddings
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4298.90it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\lenovo\AppData\Local\Temp\ipykernel_25204\2804545849.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


#### Vector DB

In [8]:
class VectorStore:
    #Manages document embeddings in a ChromaDB vector store
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 119


In [9]:
# convert the text to embeddings
texts=[doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 119 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


Generated embeddings with shape: (119, 384)
Adding 119 documents to vector store...
Successfully added 119 documents to vector store
Total documents in collection: 238


## Retriever Pipeline from VectorStore



In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [11]:
rag_retriever.retrieve("What is Web Analytics?")

Retrieving documents for query: 'What is Web Analytics?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c1c0d1e9_0',
  'content': 'WEB ANALYTICS',
  'metadata': {'creator': 'Microsoft® PowerPoint® 2016',
   'page_label': '1',
   'producer': 'Pdftools SDK',
   'doc_index': 0,
   'moddate': '2026-07-23T17:55:51+05:30',
   'content_length': 13,
   'source_file': 'n3.pdf',
   'total_pages': 54,
   'creationdate': '2025-07-07T11:00:56+00:00',
   'source': '..\\data\\pdf\\n3.pdf',
   'file_type': 'pdf',
   'page': 0},
  'similarity_score': 0.7374675273895264,
  'distance': 0.26253247261047363,
  'rank': 1},
 {'id': 'doc_5a49559f_0',
  'content': 'WEB ANALYTICS',
  'metadata': {'page': 0,
   'moddate': '2026-07-23T17:55:51+05:30',
   'file_type': 'pdf',
   'source_file': 'n3.pdf',
   'page_label': '1',
   'creationdate': '2025-07-07T11:00:56+00:00',
   'content_length': 13,
   'producer': 'Pdftools SDK',
   'doc_index': 0,
   'total_pages': 54,
   'source': '..\\data\\pdf\\n3.pdf',
   'creator': 'Microsoft® PowerPoint® 2016'},
  'similarity_score': 0.7374675273895264,
  'distance':

In [12]:
rag_retriever.retrieve("What is SEO?")

Retrieving documents for query: 'What is SEO?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c5e43336_30',
  'content': 'What It Measures:\n• Search Engine Rankings (SERP visibility): Where your site appears in\nGoogle/Bing results.\n• Backlinks & Referral Traffic: How many other websites link to yours, and how\nmuch traffic comes from them.\n• Social Media Metrics: Likes, shares, comments, impressions, and engagement\nacross platforms.\n• Mentions & Sentiment Analysis: What people are saying about your brand or\nproducts online.\n• Competitor Insights: Comparison of traffic, keywords, and marketing efforts.\n• Ad Performance: Impressions, clicks, and conversions from external ad platforms.\n07-07-2025 V IM.Sc –WEB ANALYTICS BY  NIDHIN SAJI',
  'metadata': {'producer': 'Pdftools SDK',
   'source': '..\\data\\pdf\\n3.pdf',
   'source_file': 'n3.pdf',
   'creationdate': '2025-07-07T11:00:56+00:00',
   'page_label': '27',
   'creator': 'Microsoft® PowerPoint® 2016',
   'content_length': 618,
   'total_pages': 54,
   'file_type': 'pdf',
   'page': 26,
   'moddate': '2

Integration of VectorDB context pipeline with LLM output

In [17]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [18]:
answer=rag_simple("What is web analytics?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is web analytics?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Web analytics is the systematic measurement and analysis of data about a website’s performance and its users—tracking how visitors interact, what content they engage with, and how well the site meets business goals. In e‑commerce, it focuses on evaluating strategy effectiveness, current site health, and user demographics and demands.


In [19]:
answer=rag_simple("What is SEO?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is SEO?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


SEO (Search Engine Optimization) is the practice of optimizing a website’s content, structure, and technical aspects so that search engines like Google and Bing rank it higher in their results pages (SERPs), thereby increasing organic visibility and traffic.


Enhanced RAG pipeline

In [20]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output



In [22]:
# Example usage:
result = rag_advanced("What is a Bounce rate?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:500])

Retrieving documents for query: 'What is a Bounce rate?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: A bounce rate is the percentage of visitors who leave a website after viewing only one page, without taking any further action.
Sources: [{'source': 'n3.pdf', 'page': 42, 'score': 0.5317776203155518, 'preview': "2. Bounce Rate\n• Definition:\nBounce rate is the percentage of visitors who leave the website after viewing only one \npage, without taking any further action\nWhy it's important:\nHigh bounce rate may indicate: Irrelevant content, Poor user experience, Misleading traffic \nsources\nHowever:\nA high bounce..."}, {'source': 'n3.pdf', 'page': 42, 'score': 0.5317776203155518, 'preview': "2. Bounce Rate\n• Definition:\nBounce rate is the percentage of visitors who leave the website after viewing only one \npage, without taking any further action\nWhy it's important:\nHigh bounce rate may indicate: Irrelevant content, Poor user experience, Misleading traffic \nsources\nHowever:\nA high bounce..."}]
Confidence: 0.5317776203155518
Context Preview: 2. Bounce Rate
• Definition:

In [24]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }



In [27]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Demographic info in web analytics", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Demographic info in web analytics'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
• 4. Demographic Info
• Definition:
Demographic info refers to user data such as age, gender, language, and 
location.
• Collected via tools like:
Google Analytics (when enabled with advertising features)
• Why it’s important:
• Helps tailor content, 

design, and marketing.
• Supports targeting in advertising.
• Reveals who your audience really is.
• Example:
If most users are aged 18–24 from India, you may adjust language tone, 
content topics, or ad targeting accordingly.
07-07-2025 V IM.Sc –WEB ANALYTICS BY  NIDHIN SAJI

• 4. Demographic Info
• Definition:
Demographic info refers to user data such as age, gender, language, and 
location.
• Collected via tools like:
Google Analytics (when enabled with advertising features)
• Why it’s important:
• Helps tailor content, design, and marketing.
• Supports targeting in advertising.
• Reveals who your audience really is.
• Example:
If most users are aged 18–24 from India, you may adjust language tone, 
content topics, or ad targeting accordingly.
07-07-2025 V IM.Sc –WEB ANALYTICS BY  NIDHIN SAJI

Basic web analytics metrics
•Pageviews
•Bounce Rate
•Pages per Session
•Demographic Info
•Devices
07-07-2025 V IM.Sc –WEB ANALYTICS BY  NIDHIN SAJI

Question: Demographic info in web analytics
